# 01 - Dataset Exploration (AITEX)

English: This notebook verifies the raw dataset before any model is written.
Every number below is measured from `data/raw`, not taken from the dataset
documentation. Where our copy differs from the official description, the
difference is documented rather than hidden.

Hindi: Ye notebook model likhne se pehle raw dataset verify karta hai.
Har number `data/raw` se directly measure kiya gaya hai. Jahan hamari copy
official description se alag hai, wo difference openly document kiya gaya hai.

## 1. Imports and paths

Hindi: `cv2` image read/process karne ke liye, `pathlib` folders/files ke
paths handle karne ke liye.

In [ ]:
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW = PROJECT_ROOT / "data" / "raw"
DEFECT_DIR = RAW / "Defect_images"
NODEFECT_DIR = RAW / "NODefect_images"
MASK_DIR = RAW / "Mask_images"

print("Defect folder :", DEFECT_DIR)
print("Normal folder :", NODEFECT_DIR)
print("Mask folder   :", MASK_DIR)

## 2. File counts

Hindi: Pehle confirm karte hain ki kitni images hain.

In [ ]:
defect_files = sorted(DEFECT_DIR.rglob("*.png"))
normal_files = sorted(NODEFECT_DIR.rglob("*.png"))
mask_files = sorted(MASK_DIR.rglob("*.png"))

print(f"Defect images : {len(defect_files)}")
print(f"Normal images : {len(normal_files)}")
print(f"Mask files    : {len(mask_files)}")
print(f"Classification total: {len(defect_files) + len(normal_files)}")
print("Normal fabric subfolders:", sorted({p.parent.name for p in normal_files}))

### Dataset version note

The official AITEX page describes **245** original images. This copy has
**247** classification images. The two extra files are identified in section 6
and are genuine captures, not duplicates.

Hindi: Official count 245 hai, hamari copy mein 247 hai. Do extra files genuine
captures hain, duplicates nahi. Ye difference openly document karna zaroori hai.

## 3. Defect code distribution

Hindi: Filename ke defect code se class distribution nikaalte hain.

In [ ]:
DEFECT_CLASSES = {
    "000": "Normal", "002": "Broken End", "006": "Broken Yarn",
    "010": "Broken Pick", "016": "Weft Curling", "019": "Fuzzyball",
    "022": "Cut Selvage", "023": "Crease", "025": "Warp Ball",
    "027": "Knots", "029": "Contamination", "030": "Nep",
    "036": "Weft Crack",
}

def code_of(path):
    parts = path.stem.split("_")
    return parts[1].zfill(3)

counter = Counter(code_of(p) for p in defect_files)
counter["000"] = len(normal_files)

table = pd.DataFrame(
    [(code, DEFECT_CLASSES[code], n) for code, n in sorted(counter.items())],
    columns=["code", "class", "images"],
)
table["share_pct"] = (table["images"] / table["images"].sum() * 100).round(2)
table

In [ ]:
figure, axis = plt.subplots(figsize=(10, 4))
ordered = table[table["code"] != "000"].sort_values("images")
axis.barh(ordered["class"], ordered["images"], color="#4C72B0")
axis.set_title("Original images per defect class")
axis.set_xlabel("number of original images")
for index, value in enumerate(ordered["images"]):
    axis.text(value + 0.2, index, str(value), va="center")
figure.tight_layout()
plt.show()

print("Knots, Contamination and Weft Crack have exactly one image each.")
print("Hindi: In teen classes ka sirf ek-ek image hai.")

## 4. Mask pairing

Three irregularities exist here and each one changes the code we need to write.

Hindi: Mask pairing mein teen problems hain, aur har ek ka code alag hoga.

In [ ]:
by_base = {}
for path in mask_files:
    by_base.setdefault("_".join(path.stem.split("_")[:3]), []).append(path.name)

multi = {base: names for base, names in by_base.items() if len(names) > 1}
print(f"Unique images with masks: {len(by_base)}")
print("Images with MORE THAN ONE mask file:")
for base, names in multi.items():
    print(f"  {base} -> {names}")

missing = sorted({p.stem for p in defect_files} - set(by_base))
print("Defect images WITHOUT any mask:", missing)

In [ ]:
empty = []
for base in by_base:
    total = 0
    for name in by_base[base]:
        mask = cv2.imread(str(MASK_DIR / name), cv2.IMREAD_GRAYSCALE)
        total += int((mask > 127).sum())
    if total == 0:
        empty.append(base)
print("Images whose mask has ZERO defect pixels:", empty)
print("Hindi: In images ke mask bilkul khali hain, inhe segmentation se")
print("exclude karna zaroori hai warna model kuch galat seekhega.")

In [ ]:
sample = cv2.imread(str(MASK_DIR / by_base['0010_006_02'][0]), cv2.IMREAD_GRAYSCALE)
values = np.unique(sample)
print("Number of distinct grey values in a mask:", len(values))
print("Values:", values[:20], "..." if len(values) > 20 else "")
print("Pixels > 0 :", int((sample > 0).sum()))
print("Pixels > 127:", int((sample > 127).sum()))
print()
print("Masks are antialiased grayscale, not binary.")
print("Hindi: Mask binary nahi hai, usme antialiasing values hain.")
print("Isliye threshold 127 use karna zaroori hai, '> 0' se lagbhag poori")
print("image defective mark ho jayegi.")

## 5. Image dimensions and mask area

Hindi: Image dimensions aur mask area check karte hain. Ye batata hai ki
patch size kyun zaroori hai.

In [ ]:
dims = Counter()
for path in defect_files + normal_files:
    dims[cv2.imread(str(path)).shape[:2]] += 1
print("Image dimensions (height, width):")
for shape, count in dims.most_common():
    print(f"  {shape}  ->  {count} images")

odd = [p.name for p in defect_files if cv2.imread(str(p)).shape[:2] != (256, 4096)]
print("Images that are not 256x4096:", odd)
print("Hindi: Ye image 3796 px wide hai, isliye patch extraction mein")
print("remainder ko pad karna padega warna last defect region lost ho jayega.")

In [ ]:
areas = []
for base, names in by_base.items():
    image = cv2.imread(str(DEFECT_DIR / f"{base}.png"), cv2.IMREAD_GRAYSCALE)
    if image is None:
        continue
    total = sum(int((cv2.imread(str(MASK_DIR / n), cv2.IMREAD_GRAYSCALE) > 127).sum()) for n in names)
    areas.append(total / image.size)

areas = np.array(areas)
print(f"Defect area fraction  mean   : {areas.mean() * 100:.4f}%")
print(f"Defect area fraction  median : {np.median(areas) * 100:.4f}%")
print(f"Defect area fraction  max    : {areas.max() * 100:.4f}%")
print()
print("Hindi: Defect area median sirf 0.009% hai. Matlab defect bahut chhote")
print("hain. Isliye (a) 256x256 patches zaroori hain, (b) segmentation mein")
print("sirf BCE kaafi nahi, Dice + BCE chahiye.")

## 6. Duplicate check

Hindi: Confirm karte hain ki koi duplicate image to nahi hai. Ye 247 vs 245
ke difference ko explain karne ke liye zaroori hai.

In [ ]:
import hashlib

seen = {}
duplicates = []
for path in defect_files + normal_files:
    digest = hashlib.md5(path.read_bytes()).hexdigest()
    if digest in seen:
        duplicates.append((seen[digest].name, path.name))
    else:
        seen[digest] = path

print("Byte-level duplicate pairs:", duplicates if duplicates else "none")

a = NODEFECT_DIR / "2608691-202020u" / "0018_000_01.png"
b = NODEFECT_DIR / "2608691-202020u" / "0018_00_01.png"
ia, ib = cv2.imread(str(a)), cv2.imread(str(b))
print("0018_000_01 shape:", ia.shape, " 0018_00_01 shape:", ib.shape)
print("pixel-identical:", np.array_equal(ia, ib))
print()
print("Result: 0018_00_01.png is a genuine extra capture with a malformed")
print("two-digit defect code. Together with the differently-cropped")
print("0094_027_05.png, this fully explains 247 vs 245.")
print("Hindi: Ye dono asli extra images hain, isliye 247 vs 245 ka difference")
print("poori tarah explain ho gaya.")

## 7. Visual check

Hindi: Ek defective image aur uska mask dekh lete hain.

In [ ]:
image_path = DEFECT_DIR / "0010_006_02.png"
image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
mask = cv2.imread(str(MASK_DIR / "0010_006_02_mask.png"), cv2.IMREAD_GRAYSCALE)

print("Image name :", image_path.name)
print("Image shape:", image.shape)
print("Mask shape :", mask.shape)

figure, axes = plt.subplots(3, 1, figsize=(14, 6))
axes[0].imshow(image)
axes[0].set_title(f"{image_path.name} - Broken Yarn")
axes[1].imshow(mask, cmap="gray")
axes[1].set_title("Ground truth mask")
axes[2].imshow(image)
axes[2].imshow(mask, cmap="jet", alpha=0.5)
axes[2].set_title("Overlay")
for axis in axes:
    axis.axis("off")
figure.tight_layout()
plt.show()

## 8. Patch density

Hindi: 256x4096 strip mein kitne patches bante hain, ye confirm karte hain.

In [ ]:
print(f"4096 / 256 = {4096 // 256} non-overlapping patches per image")
print(f"Approximate patch count: {(len(defect_files) + len(normal_files)) * 16}")
print()
print("But most defective patches contain no defect at all. Only patches")
print("whose mask overlaps them keep the defect label.")
print("Hindi: Har defective image ke sirf 1-2 patches mein hi defect hota hai,")
print("isliye usable training patches ki sankhya images se kafi kam hoti hai.")

## 9. Fabric roll structure and the second leakage risk

Hindi: Normal images sirf kuch physical fabric rolls se aati hain. Ye
dusra leakage risk hai.

The two-digit `ff` group in the filename is the physical fabric code. All
normal images of one roll live in a folder named after that roll.

In [ ]:
from collections import defaultdict

def parse(stem):
    parts = stem.split("_")
    return parts[0], parts[1].zfill(3), parts[2]

roll_to_fabrics = defaultdict(set)
for path in normal_files:
    _, _, fabric = parse(path.stem)
    roll_to_fabrics[path.parent.name].add(fabric)

print("Normal roll folder -> fabric code:")
for roll, fabrics in sorted(roll_to_fabrics.items()):
    print(f"  {roll} -> {sorted(fabrics)}")

rolls = defaultdict(set)
for path in defect_files:
    _, _, fabric = parse(path.stem)
    rolls[fabric].add(DEFECT_CLASSES[code_of(path)])

print()
print("Defect fabric code -> defect classes:")
for fabric, classes in sorted(rolls.items()):
    print(f"  {fabric} -> {sorted(classes)}")

all_rolls = sorted({parse(p.stem)[2] for p in defect_files + normal_files})
print()
print("Distinct fabric rolls in the dataset:", len(all_rolls), all_rolls)

### What this means for the split

There are only **8 distinct rolls**. A strictly roll-grouped split cannot be
stratified by defect class at the same time, because several defect types
exist on only one roll:

- `Knots` exists only on roll `05`
- `Weft Crack` exists only on roll `01`
- `Contamination` exists only on roll `04`

So whichever split receives roll `05` receives *all* Knots images, and the
other splits receive none.

Measured effect of both modes:

| Mode | Classes in val | Classes in test | Roll overlap |
| --- | --- | --- | --- |
| class stratified (default) | 10 | 9 | yes |
| strict roll grouped | 7 | 6 | none |

Strict grouping removes roll leakage but costs nearly half the classes in
validation and test. Both modes are available in `split_dataset.py`; the
default is class stratified so the existing experiment is not silently
changed. This limitation is real and is documented rather than hidden.

Hindi: Sirf 8 rolls hain, isliye roll-wise split aur class-wise stratification
ek saath nahi ho sakte. Strict mode leakage rokta hai par validation/test se
lagbhag aadhi classes hata deta hai.

## 10. Patch-level defect fraction and the labelling threshold

Hindi: Ye wahi numbers hain jo `DEFECT_PATCH_THRESHOLD` choose karne ke liye
chahiye.

In [ ]:
PATCH = 256

fractions = []
for path in defect_files:
    mask = None
    names = by_base.get(path.stem)
    if not names:
        continue
    total = np.zeros((256, 4096), dtype=np.uint8)
    for name in names:
        part = cv2.imread(str(MASK_DIR / name), cv2.IMREAD_GRAYSCALE) > 127
        # English: 0094_027_05 is 3796 px wide, so its mask is narrower than
        # 4096. Left-align and let the right padding cover the rest.
        # Hindi: 0094_027_05 3796 px wide hai, isliye uska mask chhota hai.
        total[:, :part.shape[1]] |= part.astype(np.uint8)
    if not total.any():
        continue
    pad_x = (PATCH - total.shape[1] % PATCH) % PATCH
    total = np.pad(total, ((0, 0), (0, pad_x)), mode="edge")
    for x in range(0, total.shape[1], PATCH):
        patch_fraction = total[:, x:x + PATCH].mean()
        if patch_fraction > 0:
            fractions.append(patch_fraction)

fractions = np.array(fractions)
print("Defect-bearing patches:", len(fractions))
for percentile in (0, 10, 25, 50, 75, 90, 100):
    print(f"  p{percentile:<3} {np.percentile(fractions, percentile):.6f}")

print()
print("Patch count if the threshold were:")
for threshold in (0.0, 0.00005, 0.0002, 0.0005, 0.001):
    print(f"  {threshold:<9} -> {int((fractions >= threshold).sum())} patches")

print()
print("Chosen threshold 0.0002 is about 13 defect pixels in a 256x256 patch.")
print("Hindi: 0.0002 ka matlab hai 256x256 patch mein ~13 defect pixels.")

## Conclusions

1. Counts verified: 106 defect, 141 normal, 247 classification, 107 mask
   files covering 105 images.
2. The 247 vs 245 difference is fully explained by two genuine extra captures.
3. `0044_019_04` and `0097_030_03` each have two mask files and must be merged.
4. `0100_025_08` has no mask, so it is classification-only.
5. `0106_010_03` has an entirely empty mask, so it is excluded from segmentation.
6. Masks are antialiased, so thresholding must be at 127.
7. `0094_027_05` is 3796 px wide, so patch extraction must pad the remainder.
8. Three classes have a single original image, so they cannot be split across
   train, validation and test. Their test scores are undefined and must be
   reported as such.
9. Median defect area is 0.009%, which drives the patch size and the choice of
   a Dice + BCE segmentation loss.
10. There are only 8 distinct fabric rolls. Strict roll grouping removes the
    leakage but cuts validation from 10 classes to 7 and test from 9 to 6, so
    class stratified splitting remains the default and the trade-off is
    documented.
11. Patch-level defect fraction has p10 = 0.000229, so a labelling threshold
    of 0.0002 keeps a genuine defect while dropping antialiasing. This is the
    measured basis of `DEFECT_PATCH_THRESHOLD`.

Hindi: In conclusions ke baad hi `python src/split_dataset.py` chalana chahiye.
Split original images pehle banega, patches uske baad. Isi se data leakage nahi hoga.